# RL basics: verifiable rewards, haiku edition

This tutorial uses Qwen3-4B and haiku poems to introduce the
**verifiable reward** pattern that underpins RL post-training:

1. Serve the base model.
2. Define a scoring function with a verifiable reward (syllable structure).
3. Evaluate the base model against that scorer.
4. GRPO-train the model with slime using the reward function.
5. Serve the trained checkpoint.
6. Evaluate it with the same scorer and compare.

Why haikus? A haiku has two attributes you can score
automatically — whether it follows the 5-7-5 syllable format
(deterministic, cheap) and whether the poem is actually good
(subjective, needs an LLM judge). That split between
*verifiable* and *subjective* rewards is exactly the landscape
RL post-training operates in. This tutorial covers the
verifiable half; see
[`003_slime_with_llm_as_judge`](../003_slime_with_llm_as_judge/)
for the LLM-judge extension.

The training config is intentionally tiny: 1×H100 and one
training iteration. It is a smoke run for the workflow, not a
quality run.

In [23]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

/home/ec2-user/training-gym/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [24]:
import re
from typing import Any

import modal

from modal_training_gym.common.dataset import HuggingFaceDataset
from modal_training_gym.common.deployment import DeploymentConfig
from modal_training_gym.common.eval import EvalConfig, EvalRowResult
from modal_training_gym.common.models import Qwen3_4B
from modal_training_gym.common.train import TrainConfig
from modal_training_gym.common.train_result import TrainResult
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.train_recipes.slime_recipe import SlimeRecipe
from modal_training_gym.train_recipes.slime_recipe.recipe import DATA_PATH

## Serve the base model

So, how does Qwen3-4B currently fare at writing haikus? We can
serve the base model and find out.

`DeploymentConfig.serve()` builds and deploys a vLLM app, then
returns a `ModelDeployment` with the concrete endpoint URL.

In [25]:
base_model = Qwen3_4B()
base_deployment = DeploymentConfig(
    model=base_model,
    app_name="qwen3-4b-base-serve",
    served_model_name="qwen3-4b-base",
).serve()
print(base_deployment.url)

https://modal-labs-joy-dev--qwen3-4b-base-serve-serve.modal.run


Now that the model has come alive, we can request it to write a haiku about a topic.

In [ ]:
response = base_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

Okay, how do we evaluate if that was a good haiku or not?
In addition to the poetic quality, haikus must follow the 5-7-5 syllable format.
We can use a simple heuristic to count the syllables in the response.

In [26]:
def _count_syllables(text: str) -> int:
    words = re.findall(r"[a-zA-Z]+", text)
    total = 0
    for word in words:
        count = len(re.findall(r"[aeiouy]+", word.lower()))
        if word.lower().endswith("e") and count > 1:
            count -= 1
        total += max(count, 1)
    return total

In [ ]:
def score_haiku(response: str) -> bool:
    lines = [line.strip() for line in response.strip().split("\n") if line.strip()]
    has_three_lines = len(lines) == 3

    syllable_counts = []
    has_passed = True
    if has_three_lines:
        for i, (line, target) in enumerate(zip(lines, [5, 7, 5])):
            count = _count_syllables(line)
            syllable_counts.append(count)
            diff = abs(count - target)
            if diff != 0:
                print(f"Line {i+1} has {count} syllables, expected {target}. Difference: {diff}")
                has_passed = False

    return has_passed

response = base_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)
print(score_haiku(response))

Seems straightforward enough, right? How do we run an eval on our base model?
We can transform our scoring function above into an Eval Configuration.

First, to explain, an Eval Configuration is a class that owns the model-calling loop.
The task-specific part is a scoring function passed to `.evaluate(...)`, which must
return `EvalRowResult`.

Before, our score_haiku function only returned a boolean, which gives 0 or 1 score.
But we can make it return more granular result on *how much off* it was from the syllable
target.

In [27]:
def score_haiku(response: str) -> EvalRowResult:
    lines = [line.strip() for line in response.strip().split("\n") if line.strip()]
    has_three_lines = len(lines) == 3

    syllable_score = 0.0
    syllable_counts = []
    if has_three_lines:
        for line, target in zip(lines, [5, 7, 5]):
            count = _count_syllables(line)
            syllable_counts.append(count)
            diff = abs(count - target)
            if diff == 0:
                syllable_score += 1.0
            elif diff == 1:
                syllable_score += 0.5
        syllable_score /= 3.0

    score = (0.25 if has_three_lines else 0.0) + 0.75 * syllable_score
    return EvalRowResult(
        score=score,
        passed=score >= 0.75,
        response=response,
        metadata={
            "lines": len(lines),
            "syllable_counts": syllable_counts,
        },
    )

In [28]:
class HaikuEvalConfig(EvalConfig):        
    def build_prompt(self, example: dict[str, Any]) -> str:
        return (
            f"Write a haiku about {example['keywords'].lower()}.\n\n"
            "Output only the three lines of the haiku, nothing else."
        )

Let's also define a Haiku dataset.
Here, we use the statworx/haiku dataset from HuggingFace.
Each row has a `keywords` topic and a reference `text` haiku.
We can use this dataset to train our model.

In [29]:
class HaikuDataset(HuggingFaceDataset):
    hf_repo = "statworx/haiku"
    input_column = "keywords"
    output_column = "text"
    output_format = "jsonl"
    apply_chat_template = False
    system_prompt = (
        "You are a haiku poet. Write a haiku about the given topic. "
        "Use the 5-7-5 syllable format across three lines."
    )
    prompt_template = "Write a haiku about {input}."

train_dataset = HaikuDataset(
    prompt_data=f"{DATA_PATH}/haiku/train.jsonl",
    eval_prompt_data=["haiku", f"{DATA_PATH}/haiku/eval.jsonl"],
    n_rows=50,
)
eval_dataset = HaikuDataset(prompt_data=f"{DATA_PATH}/haiku/eval.jsonl", n_rows=10)

Let's take a look at the eval set. Each row has a `keywords`
topic and a reference `text` haiku.

In [30]:
df = eval_dataset.to_pandas()
print(len(df))
df.head(5)

10


,source,text,text_phonemes,keywords,keyword_phonemes,gruen_score,text_punc
0,bfbarry,Delicate savage. / You'll never hold the cinde...,deh|lax|kaxt sae|vaxjh / yuwl neh|ver hhowld d...,cinder,sihn|der,0.639071,NaN
1,bfbarry,A splash and a cry. / Words pulled from the ri...,ax splaesh aend ax kray / werdz puhld frahm dh...,the riverside,dhax rih|ver|sayd,0.563353,NaN
2,bfbarry,"Steamy, mist rising. / Rocks receiving downwar...",stiy|miy mihst ray|zaxng / raaks rax|siy|vaxng...,mist rising,mihst ray|zaxng,0.538326,NaN
3,bfbarry,You were broken glass. / But I touched you eve...,yuw wer brow|kaxn glaes / baht ay tahcht yuw i...,broken glass,brow|kaxn glaes,0.703446,NaN
4,bfbarry,Eyes dance with firelight. / The Moon and I ar...,ayz daens wihdh faxr|layt / dhax muwn aend ay ...,eyes dance,ayz daens,0.830985,NaN


## Evaluate the base model

In [ ]:
base_eval = HaikuEvalConfig(
    deployment=base_deployment,
    dataset=eval_dataset,
    eval_fn=(lambda _df_row, response: score_haiku(response)),
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
).evaluate()
print(f"Base haiku score: {base_eval.accuracy:.1%}")
print(f"Passed (score >= 0.75): {base_eval.correct}/{base_eval.total}")

## Train with slime

Now, let's actually train the model to write good haikus.
Here, we use the slime framework (https://github.com/THUDM/slime) on Modal.

In [31]:
from modal_training_gym.common.haiku_reward import haiku_rm

my_training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        wandb=WandbConfig(project="slime-grpo", group="qwen3-0.6b-haiku"),

        custom_rm_function=haiku_rm,

        num_rollout=10,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_run_commands=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)

## Build and run

In [32]:
app = my_training_run.build_app()

In [ ]:
with modal.enable_output():
    with app.run():
        app.train.remote()

## Serve and evaluate the trained checkpoint

After training completes, `TrainResult.load(...)` reconstructs the
trained model with its checkpoint path and volume metadata attached.
Wrapping it in `DeploymentConfig` and calling `.serve()` deploys
that checkpoint behind vLLM.

In [ ]:
result = TrainResult.load("qwen3-4b-haiku")
print(result.latest_checkpoint_path())

deployment = DeploymentConfig(
    model=result.model,
    app_name="qwen3-4b-haiku-serve",
    served_model_name="qwen3-4b-haiku",
).serve()
print(deployment.url)

trained_eval = HaikuEvalConfig(
    deployment=deployment,
    dataset=eval_dataset,
    eval_fn=(lambda golden_df_row, response: score_haiku(response)),
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
).evaluate(debug=True)
print(f"Trained haiku score: {trained_eval.accuracy:.1%}")
print(f"Passed (score >= 0.75): {trained_eval.correct}/{trained_eval.total}")
print(f"Delta: {trained_eval.accuracy - base_eval.accuracy:+.1%}")

## What's next: RL with LLM--as-a-judge